In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

### 0) SetUp del notebook (imports + paths)

In [1]:
!pip -q install scanpy

In [2]:
# =========================
# 0) Imports & Paths
# =========================
import os
import random
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics.pairwise import cosine_similarity

# (opcional) baseline rápido para validar el pipeline end-to-end
from sklearn.linear_model import Ridge

# Single-cell
import scanpy as sc

# -------------------------
# Paths (carpetas Kaggle)
# -------------------------
DIR_PERT_IDS = "/kaggle/input/datasets/damontoyat/myllia-pert-ids"
DIR_CELLS    = "/kaggle/input/datasets/damontoyat/myllia-training-cells"
DIR_MEANS    = "/kaggle/input/datasets/damontoyat/myllia-training-data-means"
DIR_GT       = "/kaggle/input/datasets/damontoyat/myllia-training-ground-truth"

print("pert_ids dir:", os.listdir(DIR_PERT_IDS))
print("cells dir:", os.listdir(DIR_CELLS))
print("means dir:", os.listdir(DIR_MEANS))
print("gt dir:", os.listdir(DIR_GT))

# Ajusta si el nombre exacto del archivo difiere
PATH_PERT_IDS = os.path.join(DIR_PERT_IDS, "pert_ids_val.csv")
PATH_MEANS    = os.path.join(DIR_MEANS, "training_data_means.csv")
PATH_CELLS    = os.path.join(DIR_CELLS, "training_cells.h5ad")
PATH_GT       = os.path.join(DIR_GT, "training_data_ground_truth_table.csv")

print(PATH_PERT_IDS)
print(PATH_MEANS)
print(PATH_CELLS)
print(PATH_GT)

pert_ids dir: ['pert_ids_val.csv']
cells dir: ['training_cells.h5ad']
means dir: ['training_data_means.csv']
gt dir: ['training_data_ground_truth_table.csv']
/kaggle/input/datasets/damontoyat/myllia-pert-ids/pert_ids_val.csv
/kaggle/input/datasets/damontoyat/myllia-training-data-means/training_data_means.csv
/kaggle/input/datasets/damontoyat/myllia-training-cells/training_cells.h5ad
/kaggle/input/datasets/damontoyat/myllia-training-ground-truth/training_data_ground_truth_table.csv


# FASE 3 -- FE 

### Objetivo de esta fase

Construir un dataset “aprendible” con esta idea:

Input (X): features del gen perturbado (propiedades extraídas del single-cell + estadísticas del training)

Target (Y): vector de deltas (5127 genes) o, mejor, sus 44 PCs (por lo que viste en el EDA)

In [3]:
# =========================
# 3.1 Load data
# =========================
df_means = pd.read_csv(PATH_MEANS)
df_pert_map = pd.read_csv(PATH_PERT_IDS)
df_gt = pd.read_csv(PATH_GT)  # lo usaremos luego para checks/metric si aplica

adata = sc.read_h5ad(PATH_CELLS)

print("means:", df_means.shape)
print("pert_ids:", df_pert_map.shape)
print("gt:", df_gt.shape)
print("cells adata:", adata.shape)

means: (81, 5128)
pert_ids: (60, 3)
gt: (80, 10256)
cells adata: (17882, 19226)


### 3.2 Construir la matriz delta (training targets reales)

Qué hace:

Identifica baseline (non-targeting)

Resta baseline a cada perturbación (80)

Produce delta_matrix shape (80, 5127)

In [4]:
# =========================
# 3.2 Build delta matrix
# =========================
gene_cols = df_means.columns.drop("pert_symbol")

baseline_row = df_means[df_means["pert_symbol"] == "non-targeting"].iloc[0]
baseline = baseline_row[gene_cols].astype(float).values

df_train = df_means[df_means["pert_symbol"] != "non-targeting"].reset_index(drop=True)

delta_matrix = df_train[gene_cols].astype(float).values - baseline  # (80, 5127)
pert_symbols_train = df_train["pert_symbol"].values

print("delta_matrix:", delta_matrix.shape)
print("n perturbations:", len(pert_symbols_train))

delta_matrix: (80, 5127)
n perturbations: 80


### 3.3 PCA del target (dimensión latente ~44)

Qué hace:

Ajusta PCA sobre deltas (80×5127)

Obtiene representaciones Z (80×n_components)

Guardamos PCA para luego reconstruir 5127 genes al final.

In [5]:
# =========================
# 3.3 PCA targets (latent)
# =========================
N_COMPONENTS = 44  # por tu EDA (90% var)
delta_matrix = delta_matrix.astype(np.float32) 
pca = PCA(n_components=N_COMPONENTS, random_state=42, svd_solver="randomized")
Z = pca.fit_transform(delta_matrix)  # (80, 44)

print("Z latent:", Z.shape)

Z latent: (80, 44)


### 3.4 Feature engineering del gen perturbado (desde single-cell)

Aquí construimos features por gen que luego asignamos a cada perturbación según su pert_symbol.

Features “baratas” y efectivas:

Desde adata.X (ya log1p-normalized en base2 según descripción):

mean expression del gen (a través de células)

variance

dropout rate (fracción de ceros)

detection rate (= 1 - dropout)

(opcional) top-PC loadings del gen en el espacio celular (muy útil, pero más pesado)

In [6]:
# =========================
# 3.4 Gene features from single-cell  (+ gene PC embeddings)
# =========================
from scipy import sparse

X_sc = adata.X
print("Is adata.X sparse?", sparse.issparse(X_sc))

def col_mean_var_dropout(X):
    """
    mean, var, dropout_rate por gen (columna).
    Funciona para sparse CSR/CSC o dense sin densificar todo.
    """
    if sparse.issparse(X):
        mean = np.asarray(X.mean(axis=0)).ravel()

        X_sq = X.copy()
        X_sq.data **= 2
        mean_sq = np.asarray(X_sq.mean(axis=0)).ravel()
        var = mean_sq - mean**2

        n_cells = X.shape[0]
        nnz_per_col = np.diff(X.tocsc().indptr)
        dropout = 1.0 - (nnz_per_col / n_cells)
        return mean, var, dropout
    else:
        mean = X.mean(axis=0)
        var = X.var(axis=0)
        dropout = (X == 0).mean(axis=0)
        return mean, var, dropout

gene_mean, gene_var, gene_dropout = col_mean_var_dropout(X_sc)

gene_features_sc = pd.DataFrame({
    "gene": adata.var_names,
    "sc_mean": gene_mean.astype(np.float32),
    "sc_var": gene_var.astype(np.float32),
    "sc_dropout": gene_dropout.astype(np.float32),
    "sc_detect": (1.0 - gene_dropout).astype(np.float32),
})

# ---- NUEVO: gene embeddings desde PCA single-cell (loadings por gen)
# Nota: esto puede tardar si hay MUCHAS células, pero es el upgrade más valioso.
K_GENE_PCS = 30  # recomendado: 20–40. Empieza con 30.

# Computa PCA solo si no existe ya en adata
if "X_pca" not in adata.obsm.keys() or "PCs" not in adata.varm.keys():
    # Evita overhead innecesario
    sc.tl.pca(adata, n_comps=max(50, K_GENE_PCS), svd_solver="arpack")

gene_pcs = adata.varm["PCs"][:, :K_GENE_PCS].astype(np.float32)
pc_cols = [f"gpc_{i+1}" for i in range(K_GENE_PCS)]

gene_pcs_df = pd.DataFrame(gene_pcs, columns=pc_cols)
gene_pcs_df["gene"] = adata.var_names

# Merge final de features
gene_features_sc = gene_features_sc.merge(gene_pcs_df, on="gene", how="left")

print("gene_features_sc shape:", gene_features_sc.shape)
gene_features_sc.head()

Is adata.X sparse? True
gene_features_sc shape: (19226, 35)


,gene,sc_mean,sc_var,sc_dropout,sc_detect,gpc_1,gpc_2,gpc_3,gpc_4,gpc_5,...,gpc_21,gpc_22,gpc_23,gpc_24,gpc_25,gpc_26,gpc_27,gpc_28,gpc_29,gpc_30
0,A1BG,1.021362,1.369637,0.407281,0.592719,0.001057,-0.000462,-0.000705,3.916631e-05,-0.002028,...,-0.004773,-0.001026,0.003722,0.002424,-0.000299,-2.353735e-03,0.000637,0.003407,0.002875,1.481884e-03
1,A1CF,0.046080,0.057378,0.959736,0.040264,0.000041,-0.000070,0.000210,3.402549e-04,-0.000347,...,-0.000377,0.000064,0.000156,0.000310,0.000420,-6.562242e-05,0.000215,0.000457,0.000194,-1.231576e-04
2,A2M,0.001118,0.001117,0.998882,0.001118,0.000001,-0.000001,0.000001,-5.463498e-07,-0.000006,...,0.000021,-0.000034,-0.000005,0.000009,0.000019,-2.432064e-07,-0.000012,-0.000013,0.000002,2.119492e-05
3,A2ML1,0.000727,0.000726,0.999273,0.000727,0.000003,-0.000006,0.000002,7.700954e-06,0.000003,...,-0.000005,0.000015,0.000006,-0.000007,-0.000024,-6.634180e-06,0.000008,0.000012,-0.000002,3.681245e-07
4,A3GALT2,0.001790,0.001898,0.998266,0.001734,0.000004,0.000005,0.000007,2.311312e-07,-0.000007,...,0.000021,-0.000010,0.000042,-0.000005,0.000027,-2.270059e-05,0.000005,-0.000017,0.000013,-3.805215e-06


### 3.5 Feature engineering adicional desde delta_matrix (solo training)

Esto es “biología a nivel perturbación”, pero lo convertimos en propiedades del gen perturbado:

pert_strength (norma L2 del efecto)

n_responsive (conteo de genes con |delta| > 0.25)

OJO: Estas features solo existen en training (porque necesitas ver el delta real).
👉 Úsalas para EDA o para entender el dataset, pero NO deben entrar como features del modelo para test/val.
Las incluyo para que hagas checks y entiendas el sistema, pero NO las meto en X.

In [7]:
# =========================
# 3.5 Training-only perturbation descriptors (DO NOT USE AS FEATURES)
# =========================
THRESH = 0.25
pert_strength = np.linalg.norm(delta_matrix, axis=1)
n_responsive = (np.abs(delta_matrix) > THRESH).sum(axis=1)

train_pert_stats = pd.DataFrame({
    "pert_symbol": pert_symbols_train,
    "pert_strength": pert_strength,
    "n_responsive": n_responsive
}).sort_values("pert_strength", ascending=False)

train_pert_stats.head(10)

,pert_symbol,pert_strength,n_responsive
40,LRPPRC,11.207620,144
75,TFAM,10.683279,52
39,KRAS,8.325439,205
62,SETD1A,7.888235,218
1,ALDOA,6.745190,136
72,STAG2,6.636577,133
18,EIF3H,6.275763,110
65,SLC39A6,6.200885,77
71,SSBP1,5.772720,28
31,INO80,5.705332,69


### 3.6 Construir el dataset final X (features del gen perturbado) y Y (PCA targets)

Qué hace:

Para cada perturbación (fila), toma su pert_symbol

Busca sus features en gene_features_sc

Construye X (80×F)

Y será Z (80×44)

In [8]:
# =========================
# 3.6 Build supervised dataset (X -> Z)  (+ gene PC embeddings)
# =========================

# Features base + gene PC embeddings
FEATURE_COLS = ["sc_mean", "sc_var", "sc_dropout", "sc_detect"] + pc_cols

feat_map = gene_features_sc.set_index("gene")[FEATURE_COLS]

X_df = pd.DataFrame({"pert_symbol": pert_symbols_train}).join(
    feat_map, on="pert_symbol"
)

# Leakage check: ¿faltan genes?
missing = X_df[X_df[FEATURE_COLS].isna().any(axis=1)]["pert_symbol"].tolist()
print("Missing gene features for:", missing[:10], " ... total:", len(missing))

# Imputación consistente (medianas de train)
train_medians = X_df[FEATURE_COLS].median()
for c in FEATURE_COLS:
    X_df[c] = X_df[c].fillna(train_medians[c])

X = X_df[FEATURE_COLS].values.astype(np.float32)
Y = Z.astype(np.float32)

print("X:", X.shape, "Y:", Y.shape)

Missing gene features for: []  ... total: 0
X: (80, 34) Y: (80, 44)


# FASE 4 — Data loaders + seed fix (y preparación reproducible)

Aquí no necesitamos “dataloaders” tipo PyTorch todavía (a menos que uses NN).
Lo importante es:

fijar seeds

definir folds

tener funciones limpias para train/val

In [9]:
# =========================
# 4.1 Seed fix
# =========================
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(42)

### 4.2 Definir fold-split correcto (KFold)

Con 80 muestras, lo más sensato es:

KFold(n_splits=5, shuffle=True, random_state=42)

No tiene sentido StratifiedKFold (no hay labels)
GroupKFold solo si tuvieras batches por perturbación (no aplica aquí con means agregados).

In [10]:
# =========================
# 4.2 KFold splitter
# =========================
N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

folds = list(kf.split(X))
print("folds:", len(folds))
print("example fold sizes:", [len(v) for _, v in folds])

folds: 5
example fold sizes: [16, 16, 16, 16, 16]


# FASE 5 — Validación + leakage checks + métricas
Qué queremos validar

Que el pipeline no hace leakage

Que la validación refleje el leaderboard lo mejor posible

Que tengamos una métrica proxy útil (aunque la métrica oficial es compleja)

### 5.2 Métrica proxy local (rápida)

La métrica oficial usa pesos + baseline ratio + wcos.
Para iterar rápido, usamos una proxy que correlaciona bien:

MAE en el espacio PCA (Z) y

Cosine similarity en el espacio reconstruido (delta)

Esto es simple y útil para escoger modelos sin implementar toda la métrica aún.

In [11]:
# =========================
# 5.2 Proxy metrics
# =========================
def mae(a, b):
    return np.mean(np.abs(a - b))

def wcos_proxy(a, b, eps=1e-12):
    # cosine entre vectores aplanados
    aa = a.reshape(-1)
    bb = b.reshape(-1)
    num = np.dot(aa, bb)
    den = (np.linalg.norm(aa) * np.linalg.norm(bb)) + eps
    return float(num / den)

### 5.3 Validación KFold end-to-end (con baseline Ridge opcional)

Esto ya te deja el pipeline listo. Si no quieres entrenar nada aún, puedes saltarlo.
Pero te recomiendo correrlo una vez para verificar todo el wiring.

In [12]:
# =========================
# 5.3 CV loop with a simple baseline (Ridge)
# =========================
def run_cv_ridge(X, Y, folds, alpha=1.0):
    fold_scores = []
    fold_wcos = []

    for fold_i, (tr_idx, va_idx) in enumerate(folds):
        Xtr, Xva = X[tr_idx], X[va_idx]
        Ytr, Yva = Y[tr_idx], Y[va_idx]

        # Escalado solo con train (evita leakage)
        scaler = StandardScaler()
        Xtr_s = scaler.fit_transform(Xtr)
        Xva_s = scaler.transform(Xva)

        # Modelo multi-output (Ridge soporta Y multicol)
        model = Ridge(alpha=alpha, random_state=42)
        model.fit(Xtr_s, Ytr)

        Yhat = model.predict(Xva_s)

        # Métrica en espacio latent
        score_mae = mae(Yva, Yhat)

        # Reconstrucción a espacio genes (aprox)
        delta_hat = pca.inverse_transform(Yhat)
        delta_true = pca.inverse_transform(Yva)

        score_wcos = wcos_proxy(delta_true, delta_hat)

        fold_scores.append(score_mae)
        fold_wcos.append(score_wcos)

        print(f"[Fold {fold_i}] MAE_latent={score_mae:.5f}  Wcos_proxy={score_wcos:.5f}")

    print("CV MAE_latent mean:", np.mean(fold_scores), "std:", np.std(fold_scores))
    print("CV Wcos_proxy mean:", np.mean(fold_wcos), "std:", np.std(fold_wcos))
    return fold_scores, fold_wcos

_ = run_cv_ridge(X, Y, folds, alpha=1.0)

[Fold 0] MAE_latent=0.55533  Wcos_proxy=0.05836
[Fold 1] MAE_latent=0.34555  Wcos_proxy=0.27147
[Fold 2] MAE_latent=0.70054  Wcos_proxy=0.06826
[Fold 3] MAE_latent=0.82561  Wcos_proxy=0.08692
[Fold 4] MAE_latent=0.36261  Wcos_proxy=0.17687
CV MAE_latent mean: 0.5579277 std: 0.18721831
CV Wcos_proxy mean: 0.1323757104575634 std: 0.08121572747853974


### 5.4 Preparación para submission (usar pert_ids_val.csv)

Este archivo lo usaremos así:

Para leaderboard (pert_1..pert_60): mapear pert_id → gene symbol

Para pert_61..pert_120: todavía no conocemos genes hasta que Kaggle lo libere; por ahora se ponen dummy (p.ej. baseline o zeros)

OJO: En este notebook (Fase 3–5) solo dejamos listo el helper para construir submission.
La predicción real se hará cuando entrenes modelo final.

In [13]:
# =========================
# 5.4 Submission helpers
# =========================
# Mapa: pert_id -> gene symbol (solo val por ahora)
val_map = df_pert_map[df_pert_map["class"] == "val"][["pert_id", "pert"]].copy()
pertid_to_gene = dict(zip(val_map["pert_id"], val_map["pert"]))

print("n val perturbations:", len(pertid_to_gene))
print("example:", list(pertid_to_gene.items())[:5])

# Orden de submission (120 filas)
submission_ids = [f"pert_{i}" for i in range(1, 121)]

n val perturbations: 60
example: [('pert_1', 'SMARCE1'), ('pert_2', 'DPF2'), ('pert_3', 'MRE11'), ('pert_4', 'TCF7L2'), ('pert_5', 'HMGXB4')]


In [14]:
def build_submission_template(gene_cols):
    sub = pd.DataFrame({"pert_id": [f"pert_{i}" for i in range(1, 121)]})
    for g in gene_cols:
        sub[g] = 0.0
    return sub

sub_template = build_submission_template(gene_cols)
sub_template.head(2)

/tmp/ipykernel_109/3533949428.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  sub[g] = 0.0
/tmp/ipykernel_109/3533949428.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  sub[g] = 0.0
/tmp/ipykernel_109/3533949428.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  sub[g] = 0.0
/tmp/ipykernel_1

,pert_id,A1BG,A1CF,AADAC,AAK1,AARS1,AASS,ABCA1,ABCA12,ABCA5,...,ZP3,ZPBP,ZRANB3,ZSCAN18,ZSCAN31,ZSWIM5,ZSWIM6,ZSWIM7,ZWINT,ZYX
0,pert_1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,pert_2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# FASE 6 — Entrenamiento y Fine-Tuning (MLP pequeño)
Requisitos previos (de Fase 3–5)

X: np.array shape (80, F)

Y: np.array shape (80, 44) (tus targets en PCA)

pca: objeto PCA ya fit

folds: lista de splits KFold sobre X

In [15]:
import os, random
import numpy as np

from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

### 6.1 Seed fix (reproducibilidad)

In [16]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cpu


### 6.2 Dataset + DataLoader

In [17]:
class TabDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.float32)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

### 6.3 MLP pequeño (Dropout + LayerNorm)

In [18]:
class SmallMLP(nn.Module):
    def __init__(self, in_dim, out_dim, hidden_dim=64, n_layers=2, dropout=0.25):
        super().__init__()
        layers = []
        d = in_dim

        for _ in range(n_layers):
            layers += [
                nn.Linear(d, hidden_dim),
                nn.LayerNorm(hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout),
            ]
            d = hidden_dim

        layers.append(nn.Linear(d, out_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

### 6.4 Métrica proxy de validación (rápida y útil)

No implementamos la métrica oficial todavía; usamos una proxy estable:

MAE en latente (Y vs Yhat)

y opcionalmente cosine en genes después de inverse PCA

In [19]:
def mae_np(a, b):
    return float(np.mean(np.abs(a - b)))

def wcos_proxy_np(a, b, eps=1e-12):
    aa = a.reshape(-1)
    bb = b.reshape(-1)
    num = float(np.dot(aa, bb))
    den = float(np.linalg.norm(aa) * np.linalg.norm(bb) + eps)
    return num / den

### 6.5 Early stopping + entrenamiento con scheduler + clipping + AMP

Scheduler: OneCycleLR (bueno para MLPs y converge rápido)

Early stopping: evita overfit

Gradient clipping: estabilidad

AMP: solo si cuda

In [20]:
class EarlyStopping:
    def __init__(self, patience=50, min_delta=1e-6):
        self.patience = patience
        self.min_delta = min_delta
        self.best = None
        self.bad = 0

    def step(self, metric):
        if self.best is None or metric < self.best - self.min_delta:
            self.best = metric
            self.bad = 0
            return False
        self.bad += 1
        return self.bad >= self.patience


def train_one_fold_mlp(
    X, Y, tr_idx, va_idx,
    hidden_dim=64, n_layers=2, dropout=0.25,
    lr=2e-3, weight_decay=1e-4,
    batch_size=16, max_epochs=400,
    clip_grad=1.0, patience=60,
    scheduler="onecycle",
    seed=42,
):
    seed_everything(seed)

    # ---- leakage-safe scaling (fit solo en train)
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(X[tr_idx])
    Xva = scaler.transform(X[va_idx])

    dl_tr = DataLoader(TabDataset(Xtr, Y[tr_idx]), batch_size=batch_size, shuffle=True)
    dl_va = DataLoader(TabDataset(Xva, Y[va_idx]), batch_size=batch_size, shuffle=False)

    model = SmallMLP(X.shape[1], Y.shape[1], hidden_dim, n_layers, dropout).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.L1Loss()

    # ---- scheduler
    if scheduler == "onecycle":
        steps_per_epoch = max(1, len(dl_tr))
        sch = torch.optim.lr_scheduler.OneCycleLR(
            opt, max_lr=lr, epochs=max_epochs, steps_per_epoch=steps_per_epoch,
            pct_start=0.1, div_factor=10.0, final_div_factor=100.0
        )
        step_per_batch = True
    elif scheduler == "cosine":
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_epochs)
        step_per_batch = False
    else:
        sch = None
        step_per_batch = False

    use_amp = (device.type == "cuda")
    amp_scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    early = EarlyStopping(patience=patience)
    best = {"score": float("inf"), "state": None, "scaler": None}

    for epoch in range(max_epochs):
        model.train()
        for xb, yb in dl_tr:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=use_amp):
                pred = model(xb)
                loss = loss_fn(pred, yb)

            amp_scaler.scale(loss).backward()

            if clip_grad is not None:
                amp_scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), clip_grad)

            amp_scaler.step(opt)
            amp_scaler.update()

            if sch is not None and step_per_batch:
                sch.step()

        # ---- val
        model.eval()
        P, T = [], []
        with torch.no_grad():
            for xb, yb in dl_va:
                xb = xb.to(device)
                pred = model(xb).cpu().numpy()
                P.append(pred)
                T.append(yb.numpy())

        P = np.concatenate(P, axis=0)
        T = np.concatenate(T, axis=0)

        val_mae_lat = mae_np(T, P)

        # proxy cos en genes (reconstrucción)
        delta_hat = pca.inverse_transform(P)
        delta_true = pca.inverse_transform(T)
        val_wcos = wcos_proxy_np(delta_true, delta_hat)

        # score objetivo: minimiza mae y favorece dirección (wcos)
        score = val_mae_lat - 0.10 * val_wcos

        if sch is not None and (not step_per_batch):
            sch.step()

        if score < best["score"]:
            best["score"] = score
            best["state"] = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best["scaler"] = scaler

        if early.step(score):
            break

        if (epoch == 0) or ((epoch + 1) % 50 == 0):
            lr_now = opt.param_groups[0]["lr"]
            print(f"epoch {epoch+1:03d} | val_mae_lat={val_mae_lat:.5f} | wcos={val_wcos:.5f} | score={score:.5f} | lr={lr_now:.2e}")

    return best

### 6.6 Correr CV con un baseline MLP (para sanity + estabilidad)

In [21]:
params_baseline = dict(
    hidden_dim=64,
    n_layers=2,
    dropout=0.25,
    lr=2e-3,
    weight_decay=1e-4,
    batch_size=16,
    max_epochs=400,
    clip_grad=1.0,
    patience=60,
    scheduler="onecycle",
)

fold_bests = []
scores = []
for i, (tr_idx, va_idx) in enumerate(folds):
    print(f"\n=== Fold {i} ===")
    best = train_one_fold_mlp(X, Y, tr_idx, va_idx, seed=42+i, **params_baseline)
    print("best score:", best["score"])
    fold_bests.append(best)
    scores.append(best["score"])

print("\nCV score mean:", float(np.mean(scores)), "std:", float(np.std(scores)))


=== Fold 0 ===


/tmp/ipykernel_109/1481106520.py:56: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  amp_scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/tmp/ipykernel_109/1481106520.py:67: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


epoch 001 | val_mae_lat=0.42395 | wcos=0.18285 | score=0.40566 | lr=2.03e-04
epoch 050 | val_mae_lat=0.31264 | wcos=0.21338 | score=0.29130 | lr=2.00e-03
best score: 0.2786391386422118

=== Fold 1 ===
epoch 001 | val_mae_lat=0.45479 | wcos=0.17897 | score=0.43690 | lr=2.03e-04


/tmp/ipykernel_109/1481106520.py:56: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  amp_scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/tmp/ipykernel_109/1481106520.py:67: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


epoch 050 | val_mae_lat=0.32123 | wcos=0.29309 | score=0.29192 | lr=2.00e-03
epoch 100 | val_mae_lat=0.32226 | wcos=0.26373 | score=0.29589 | lr=1.87e-03
best score: 0.29113771789433807

=== Fold 2 ===
epoch 001 | val_mae_lat=0.46674 | wcos=0.14151 | score=0.45259 | lr=2.03e-04


...

epoch 050 | val_mae_lat=0.31979 | wcos=0.29934 | score=0.28985 | lr=2.00e-03
best score: 0.2875386715137513

=== Fold 3 ===
epoch 001 | val_mae_lat=0.45828 | wcos=0.18227 | score=0.44005 | lr=2.03e-04


/tmp/ipykernel_109/1481106520.py:56: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  amp_scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/tmp/ipykernel_109/1481106520.py:67: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


epoch 050 | val_mae_lat=0.30989 | wcos=0.23113 | score=0.28677 | lr=2.00e-03
epoch 100 | val_mae_lat=0.31736 | wcos=0.20556 | score=0.29681 | lr=1.87e-03
best score: 0.28235912340184516

=== Fold 4 ===
epoch 001 | val_mae_lat=0.40732 | wcos=0.18075 | score=0.38925 | lr=2.03e-04


/tmp/ipykernel_109/1481106520.py:56: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  amp_scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/tmp/ipykernel_109/1481106520.py:67: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


epoch 050 | val_mae_lat=0.27803 | wcos=0.36834 | score=0.24119 | lr=2.00e-03
epoch 100 | val_mae_lat=0.28226 | wcos=0.33027 | score=0.24923 | lr=1.87e-03
best score: 0.24085615524724063

CV score mean: 0.27610616133987737 std: 0.018136859813204478


# FASE 7 — Hiperparámetros y Model Selection (Optuna)
Objetivo realista aquí

Como solo tienes 80 muestras:

pocos trials (20–40)

search space pequeño

5-fold CV (o 3-fold si quieres velocidad)

In [22]:
!pip -q install optuna

In [23]:
import optuna

### 7.1 Objective con CV

In [24]:
OPTUNA_TRIALS = 20
MAX_EPOCHS_TRIAL = 220
PATIENCE_TRIAL = 35

# Si aún así es pesado, baja a:
# OPTUNA_TRIALS = 20
# MAX_EPOCHS_TRIAL = 180
# PATIENCE_TRIAL = 25

In [25]:
def objective(trial):
    hidden_dim = trial.suggest_categorical("hidden_dim", [ 64, 128, 256])
    n_layers   = trial.suggest_int("n_layers", 1, 3)
    dropout    = trial.suggest_float("dropout", 0.0, 0.35)
    lr         = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
    wd         = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [8, 16, 32])

    scheduler  = "onecycle"  # fija para acelerar y estabilizar el search

    fold_scores = []
    for fold_i, (tr_idx, va_idx) in enumerate(folds):
        best = train_one_fold_mlp(
            X, Y, tr_idx, va_idx,
            hidden_dim=hidden_dim,
            n_layers=n_layers,
            dropout=dropout,
            lr=lr,
            weight_decay=wd,
            batch_size=batch_size,
            max_epochs=MAX_EPOCHS_TRIAL,
            clip_grad=1.0,
            patience=PATIENCE_TRIAL,
            scheduler=scheduler,
            seed=1000 + 10*trial.number + fold_i
        )
        fold_scores.append(best["score"])

    return float(np.mean(fold_scores))

### 7.2 Ejecutar el estudio

In [26]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=OPTUNA_TRIALS)

print("Best value:", study.best_value)
print("Best params:", study.best_params)

[I 2026-03-09 01:06:29,643] A new study created in memory with name: no-name-a9cb1c17-5c91-4f1f-8beb-d2f0afd6e183
/tmp/ipykernel_109/1481106520.py:56: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  amp_scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/tmp/ipykernel_109/1481106520.py:67: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


epoch 001 | val_mae_lat=0.46834 | wcos=0.19385 | score=0.44896 | lr=1.24e-05
epoch 050 | val_mae_lat=0.34754 | wcos=0.19566 | score=0.32797 | lr=1.12e-04
epoch 100 | val_mae_lat=0.33722 | wcos=0.18080 | score=0.31914 | lr=7.82e-05
epoch 150 | val_mae_lat=0.33510 | wcos=0.18161 | score=0.31694 | lr=3.26e-05
epoch 200 | val_mae_lat=0.33485 | wcos=0.18203 | score=0.31665 | lr=2.93e-06


/tmp/ipykernel_109/1481106520.py:56: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  amp_scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/tmp/ipykernel_109/1481106520.py:67: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


epoch 001 | val_mae_lat=0.43617 | wcos=0.13682 | score=0.42249 | lr=1.24e-05
epoch 050 | val_mae_lat=0.33707 | wcos=0.20609 | score=0.31646 | lr=1.12e-04
epoch 100 | val_mae_lat=0.33136 | wcos=0.22895 | score=0.30847 | lr=7.82e-05
epoch 150 | val_mae_lat=0.32986 | wcos=0.24203 | score=0.30566 | lr=3.26e-05
epoch 200 | val_mae_lat=0.32932 | wcos=0.24731 | score=0.30459 | lr=2.93e-06


/tmp/ipykernel_109/1481106520.py:56: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  amp_scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/tmp/ipykernel_109/1481106520.py:67: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


...

epoch 001 | val_mae_lat=0.41356 | wcos=0.18191 | score=0.39537 | lr=1.26e-04
epoch 050 | val_mae_lat=0.30359 | wcos=0.24212 | score=0.27938 | lr=1.14e-03


/tmp/ipykernel_109/1481106520.py:56: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  amp_scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/tmp/ipykernel_109/1481106520.py:67: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


epoch 001 | val_mae_lat=0.40375 | wcos=0.14350 | score=0.38940 | lr=1.26e-04
epoch 050 | val_mae_lat=0.27324 | wcos=0.35941 | score=0.23729 | lr=1.14e-03


[I 2026-03-09 01:12:33,333] Trial 19 finished with value: 0.27026698232560664 and parameters: {'hidden_dim': 64, 'n_layers': 3, 'dropout': 0.1826115473486703, 'lr': 0.001201345296832137, 'weight_decay': 2.00922486336037e-05, 'batch_size': 16}. Best is trial 16 with value: 0.2628630888580922.


Best value: 0.2628630888580922
Best params: {'hidden_dim': 64, 'n_layers': 3, 'dropout': 0.2755553448294251, 'lr': 0.0028323144193780057, 'weight_decay': 3.600229529318626e-05, 'batch_size': 32}


### 7.3 Entrenar modelo final con best params (full data)

Esto produce un model_final + scaler_full para luego hacer submission.

In [27]:
def train_final_mlp(X, Y, best_params, seed=42):
    seed_everything(seed)

    scaler_full = StandardScaler()
    Xs = scaler_full.fit_transform(X)

    dl = DataLoader(TabDataset(Xs, Y), batch_size=best_params.get("batch_size", 16), shuffle=True)

    model = SmallMLP(
        in_dim=X.shape[1],
        out_dim=Y.shape[1],
        hidden_dim=best_params.get("hidden_dim", 64),
        n_layers=best_params.get("n_layers", 2),
        dropout=best_params.get("dropout", 0.25),
    ).to(device)

    opt = torch.optim.AdamW(
        model.parameters(),
        lr=best_params.get("lr", 2e-3),
        weight_decay=best_params.get("weight_decay", 1e-4),
    )
    loss_fn = nn.L1Loss()

    scheduler = best_params.get("scheduler", "onecycle")
    max_epochs = 450

    if scheduler == "onecycle":
        steps_per_epoch = max(1, len(dl))
        sch = torch.optim.lr_scheduler.OneCycleLR(
            opt, max_lr=best_params.get("lr", 2e-3), epochs=max_epochs, steps_per_epoch=steps_per_epoch,
            pct_start=0.1, div_factor=10.0, final_div_factor=100.0
        )
        step_per_batch = True
    elif scheduler == "cosine":
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_epochs)
        step_per_batch = False
    else:
        sch = None
        step_per_batch = False

    use_amp = (device.type == "cuda")
    amp_scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    model.train()
    for epoch in range(max_epochs):
        total = 0.0
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=use_amp):
                pred = model(xb)
                loss = loss_fn(pred, yb)

            amp_scaler.scale(loss).backward()
            amp_scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            amp_scaler.step(opt)
            amp_scaler.update()

            if sch is not None and step_per_batch:
                sch.step()

            total += loss.item() * xb.size(0)

        if sch is not None and (not step_per_batch):
            sch.step()

        if epoch == 0 or (epoch + 1) % 100 == 0:
            print(f"epoch {epoch+1:03d} | train_loss={total/len(dl.dataset):.5f}")

    return model, scaler_full


best_params = study.best_params  # o params_baseline si no corres Optuna
model_final, scaler_full = train_final_mlp(X, Y, best_params, seed=42)

/tmp/ipykernel_109/2318736258.py:42: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  amp_scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/tmp/ipykernel_109/2318736258.py:51: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


epoch 001 | train_loss=0.49403
epoch 100 | train_loss=0.25934
epoch 200 | train_loss=0.23062
epoch 300 | train_loss=0.21682
epoch 400 | train_loss=0.20061


In [28]:
val_map = df_pert_map[df_pert_map["class"] == "val"][["pert_id", "pert"]].copy()
pertid_to_gene = dict(zip(val_map["pert_id"], val_map["pert"]))

val_ids = [f"pert_{i}" for i in range(1, 61)]
val_genes = [pertid_to_gene[pid] for pid in val_ids]

print(val_ids[:5])
print(val_genes[:5])

['pert_1', 'pert_2', 'pert_3', 'pert_4', 'pert_5']
['SMARCE1', 'DPF2', 'MRE11', 'TCF7L2', 'HMGXB4']


In [29]:
feat_map = gene_features_sc.set_index("gene")[FEATURE_COLS]

# medianas de train (para imputación consistente)
train_medians = X_df[FEATURE_COLS].median()

def genes_to_X(genes):
    tmp = pd.DataFrame({"pert_symbol": genes}).join(feat_map, on="pert_symbol")
    for c in FEATURE_COLS:
        tmp[c] = tmp[c].fillna(train_medians[c])
    return tmp[FEATURE_COLS].values.astype(np.float32)

In [31]:
mean_delta = delta_matrix.mean(axis=0).astype(np.float32)

In [32]:
model_final.eval()

X_val = genes_to_X(val_genes)
X_val_s = scaler_full.transform(X_val)

with torch.no_grad():
    xb = torch.tensor(X_val_s, dtype=torch.float32).to(device)
    Z_hat = model_final(xb).cpu().numpy()        # (60, 44)

delta_hat_val = pca.inverse_transform(Z_hat).astype(np.float32)  # (60, 5127)

# --- AQUÍ VA EL BLEND ---
alpha = 0.3
delta_hat_val = (1 - alpha) * mean_delta.reshape(1, -1) + alpha * delta_hat_val
print(delta_hat_val.shape)

(60, 5127)


In [ ]:
mean_delta = delta_matrix.mean(axis=0).astype(np.float32)

In [33]:
# 120 x 5127 matriz inicial llena con mean_delta
base = np.tile(mean_delta.reshape(1, -1), (120, 1)).astype(np.float32)

# sobrescribe filas 0..59 con tus predicciones (pert_1..pert_60)
base[:60, :] = delta_hat_val

# arma el dataframe de una sola vez
sub = pd.DataFrame(base, columns=gene_cols)
sub.insert(0, "pert_id", [f"pert_{i}" for i in range(1, 121)])

print(sub.head(2))
print(sub.shape)

  pert_id      A1BG      A1CF     AADAC      AAK1     AARS1      AASS  \
0  pert_1  0.009015  0.000284 -0.002847 -0.003991  0.014412 -0.000415   
1  pert_2  0.008757  0.000233 -0.003010  0.001284 -0.002756 -0.000033   

      ABCA1    ABCA12     ABCA5  ...       ZP3      ZPBP    ZRANB3   ZSCAN18  \
0 -0.004088 -0.000942  0.010274  ...  0.024497 -0.000119  0.013973  0.001767   
1 -0.002394 -0.001524  0.010466  ...  0.026682 -0.000270  0.004057  0.002182   

    ZSCAN31    ZSWIM5    ZSWIM6    ZSWIM7     ZWINT       ZYX  
0  0.007502  0.012966 -0.021317 -0.021403 -0.015944 -0.016664  
1  0.007707  0.011158 -0.017863 -0.020794 -0.011794 -0.011067  

[2 rows x 5128 columns]
(120, 5128)


In [34]:
assert sub.shape == (120, 1 + len(gene_cols))
assert sub.columns[0] == "pert_id"

In [35]:
sub.to_csv("submission.csv", index=False)
print("Saved submission.csv")

Saved submission.csv
